# 01 · Crop-disease vision model → ExecuTorch (.pte) — v2
**Bharat AI-SoC 2026–27 · PS1 · Offline Crop-Disease/Pest Detection and Advisory**

**What changed in v2:** training uses Kaggle's own pre-installed PyTorch (nothing gets upgraded). The ExecuTorch export runs in a **separate Python environment** created in section 5. The two PyTorch versions never mix, so the `bcomplex32` error can't happen.

**Kaggle setup:**
1. *Create ▸ New Notebook ▸ File ▸ Import Notebook*, then upload this file. **Start a new notebook**; don't reuse the broken session.
2. Right panel ▸ *Session options*: **Accelerator → GPU T4 x2** (not P100: recent PyTorch builds no longer support it) and **Internet → On**.
3. Right panel ▸ *Add Input*: search **"plantvillage dataset"** (by *abdallahalidev*) and add it.
4. **Run All**. Takes about 40–50 min; section 5 spends about 5–10 min setting up the export environment.
5. Download **`ps1_vision_outputs.zip`** from the Output panel (`/kaggle/working`).

**Outputs:** `crop_disease_int8.pte` (for the app, ~5 MB) · `crop_disease_fp32.pte` · `labels.json` · `severity_config.json` · `metrics.json` · `confusion_matrix.png` · `severity_examples.png`.

**App input contract:** `float32 [1,3,224,224]`, RGB scaled to **0–1** (normalisation is inside the model) → **probabilities** for 38 classes (softmax is inside the model).

In [ ]:
# Uses Kaggle's pre-installed torch/torchvision. Do NOT pip-install torch or executorch in this kernel.
!pip install -q --default-timeout=120 --retries 10 kagglehub opencv-python-headless scikit-learn
import os, sys, json, random, glob, time, shutil, subprocess
import torch, torchvision
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Accelerator to GPU T4 x2")

In [ ]:
# ---- Config -------------------------------------------------------------
SEED, IMG_SIZE, BATCH, EPOCHS, LR = 42, 224, 64, 4, 1e-3
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK      = "/kaggle/working" if IS_KAGGLE else "/content"
OUT_DIR   = os.environ.get("OUT_DIR_OVERRIDE", f"{WORK}/out")
ENV_DIR   = os.environ.get("ET_ENV_DIR", "/tmp/et_env")    # export environment — kept out of the outputs
FAST_TEST = os.environ.get("FAST_TEST") == "1"               # local smoke test only
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1 · Dataset — PlantVillage (38 classes, 14 crops, ~54k images)
⚠️ These are lab photos on plain backgrounds, so accuracy on field photos will be lower. Say so in the report. Phase 2 adds field images (PlantDoc), pests (IP102) and rice.

In [ ]:
if FAST_TEST:
    DATA_DIR = os.environ["FAST_DATA_DIR"]
else:
    cands = [p for p in glob.glob("/kaggle/input/**/color", recursive=True) if os.path.isdir(p)]
    if not cands:   # dataset not attached as Input → download it
        import kagglehub
        root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
        cands = [p for p in glob.glob(os.path.join(root, "**", "color"), recursive=True) if os.path.isdir(p)]
    assert cands, "PlantVillage 'color' folder not found — add the dataset via Add Input"
    DATA_DIR = cands[0]
print("DATA_DIR:", DATA_DIR)

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
def parse_label(raw):
    crop, _, cond = raw.partition("___")
    clean = lambda s: s.replace("_", " ").replace(",", "").strip()
    return {"raw": raw, "crop": clean(crop), "condition": clean(cond) or "unknown", "healthy": cond.lower() == "healthy"}
labels = [parse_label(c) for c in classes]
json.dump(labels, open(f"{OUT_DIR}/labels.json", "w"), indent=1)
print(len(classes), "classes")

In [ ]:
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

train_tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor()])
eval_tf  = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor()])
base_train, base_eval = ImageFolder(DATA_DIR, transform=train_tf), ImageFolder(DATA_DIR, transform=eval_tf)
assert base_train.classes == classes

idx = list(range(len(base_train))); random.Random(SEED).shuffle(idx)
n_val = n_test = int(0.1 * len(idx))
val_idx, test_idx, train_idx = idx[:n_val], idx[n_val:n_val + n_test], idx[n_val + n_test:]
json.dump({"data_dir": DATA_DIR, "classes": classes, "train": train_idx, "val": val_idx, "test": test_idx},
          open(f"{OUT_DIR}/splits.json", "w"))
workers = 0 if FAST_TEST else 4
train_dl = DataLoader(Subset(base_train, train_idx), BATCH, shuffle=True, num_workers=workers, pin_memory=True)
val_dl   = DataLoader(Subset(base_eval, val_idx),  BATCH, num_workers=workers)
test_dl  = DataLoader(Subset(base_eval, test_idx), BATCH, num_workers=workers)
print(f"train {len(train_idx)} · val {len(val_idx)} · test {len(test_idx)}")

## 2 · Model — MobileNetV3-Large (ImageNet-pretrained) with a 38-class head
The model is defined in `vision_model.py`, which **both** the training kernel and the export environment import, so the architecture is guaranteed to match.

In [ ]:
MODEL_SRC = """
import torch, torch.nn as nn, torchvision
class Classifier(nn.Module):
    # Normalisation + backbone + (softmax when export_mode) -> the .pte takes 0-1 RGB and returns probabilities.
    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        w = torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        self.net = torchvision.models.mobilenet_v3_large(weights=w)
        self.net.classifier[3] = nn.Linear(self.net.classifier[3].in_features, n_classes)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.export_mode = False
    def forward(self, x):
        logits = self.net((x - self.mean) / self.std)
        return torch.softmax(logits, dim=1) if self.export_mode else logits
"""
open(f"{WORK}/vision_model.py", "w").write(MODEL_SRC)
sys.path.insert(0, WORK)
from vision_model import Classifier
model = Classifier(len(classes), pretrained=not FAST_TEST).to(device)

In [ ]:
import torch.nn as nn
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl))
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def evaluate(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append(model(x.to(device)).argmax(1).cpu()); Y.append(y)
    P, Y = torch.cat(P), torch.cat(Y)
    return (P == Y).float().mean().item(), P, Y

best = -1.0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * x.size(0)
    val_acc, _, _ = evaluate(val_dl)
    print(f"epoch {ep+1}/{EPOCHS} · loss {tot/len(train_idx):.4f} · val acc {val_acc:.4f} · {time.time()-t0:.0f}s")
    if val_acc >= best:
        best = val_acc; torch.save(model.state_dict(), f"{OUT_DIR}/best.pt")
model.load_state_dict(torch.load(f"{OUT_DIR}/best.pt", map_location=device))

## 3 · Test-set metrics (for the technical report)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt, numpy as np
test_acc, P, Y = evaluate(test_dl)
macro_f1 = f1_score(Y, P, average="macro")
print(f"TEST accuracy {test_acc:.4f} · macro-F1 {macro_f1:.4f}")
report = classification_report(Y, P, labels=list(range(len(classes))), target_names=classes, output_dict=True, zero_division=0)
cm = confusion_matrix(Y, P, labels=list(range(len(classes))))
plt.figure(figsize=(12, 10)); plt.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues")
plt.title("Confusion matrix (row-normalised)"); plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150); plt.close()
metrics = {"test_accuracy_pytorch": test_acc, "macro_f1_pytorch": macro_f1, "n_test": len(test_idx), "per_class": report}
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1, default=float)

## 4 · Leaf → spots → severity (same algorithm as the app's `LeafAnalysis.kt`)
1. **Find the leaf:** measure the background colour on the photo border (Lab colour space), keep everything clearly different from it, keep the main piece(s), and fill holes so spots inside the leaf stay "leaf".
2. **Find the spots:** use the leaf's own healthy colour as the reference and mark leaf pixels that differ strongly from it. The leaf rim, shadows and background-like pixels are never counted.
3. **Severity** = spots ÷ leaf: **< 10 % Mild · 10–25 % Moderate · > 25 % Severe**.

In [ ]:
import cv2
SEV = dict(BG_DIST=18, LESION_DIST=22, BORDER=6, KEEP_PIECE=0.15, MILD=0.10, MODERATE=0.25)

def _lab(im):  # L 0–100, a/b centred — same scale as the app
    return cv2.cvtColor(im, cv2.COLOR_RGB2LAB).astype(np.float32) * np.array([100/255, 1, 1], np.float32) - np.array([0, 128, 128], np.float32)

def _fill_holes(m):
    h, w = m.shape; ff = (~m).astype(np.uint8); mask = np.zeros((h + 2, w + 2), np.uint8)
    for x in range(w):
        for y in (0, h - 1):
            if ff[y, x] == 1: cv2.floodFill(ff, mask, (x, y), 2)
    for y in range(h):
        for x in (0, w - 1):
            if ff[y, x] == 1: cv2.floodFill(ff, mask, (x, y), 2)
    return m | (ff == 1)

def severity(rgb224, cfg=SEV):
    """rgb224: 224×224×3 RGB uint8. Returns (fraction, label, lesion_mask, leaf_mask)."""
    L = _lab(rgb224); h, w = L.shape[:2]; b = cfg["BORDER"]
    ring = np.zeros((h, w), bool); ring[:b] = ring[-b:] = True; ring[:, :b] = ring[:, -b:] = True
    bg = np.median(L[ring], 0); d_bg = np.sqrt(((L - bg) ** 2).sum(-1))
    fg = (d_bg > cfg["BG_DIST"]).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(fg, 8)
    if n <= 1:
        z = np.zeros((h, w), bool); return 0.0, "No leaf", z, z
    big = st[1:, cv2.CC_STAT_AREA].max()
    keep = np.isin(lab, [i for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= cfg["KEEP_PIECE"] * big])
    leaf = _fill_holes(keep)
    if leaf.mean() > 0.97: leaf = np.ones((h, w), bool)
    core = cv2.erode(leaf.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
    hsv = cv2.cvtColor(rgb224, cv2.COLOR_RGB2HSV); H, S, V = [hsv[..., i].astype(int) for i in range(3)]
    greenish = core & (H >= 30) & (H <= 90) & (S >= 35)
    ref = np.median(L[greenish], 0) if greenish.sum() > 50 else np.median(L[core], 0)
    shadow = ((V < 110) & (S < 55)) | (V < 45)
    les = core & (np.sqrt(((L - ref) ** 2).sum(-1)) > cfg["LESION_DIST"]) & (d_bg >= cfg["BG_DIST"]) & ~shadow
    les = cv2.morphologyEx(les.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8)).astype(bool)
    frac = float(les.sum()) / max(int(core.sum()), 1)
    label = "Mild" if frac < cfg["MILD"] else "Moderate" if frac < cfg["MODERATE"] else "Severe"
    return frac, label, les, leaf

def _square224(path):
    im = np.array(Image.open(path).convert("RGB")); s = min(im.shape[:2])
    y0 = (im.shape[0] - s) // 2; x0 = (im.shape[1] - s) // 2
    return cv2.resize(cv2.resize(im[y0:y0 + s, x0:x0 + s], (512, 512)), (224, 224))

from PIL import Image
diseased = [i for i in test_idx if not labels[base_eval.targets[i]]["healthy"]][:4]
fig, ax = plt.subplots(3, max(len(diseased), 1), figsize=(4 * max(len(diseased), 1), 12), squeeze=False)
for k, i in enumerate(diseased):
    img = _square224(base_eval.samples[i][0])
    frac, lab_, les, leaf = severity(img)
    grey = img.copy(); grey[~leaf] = (0.35 * grey[~leaf] + 0.65 * 230).astype(np.uint8)
    red = img.copy(); red[les] = (0.3 * red[les] + np.array([0.7 * 255, 0, 0])).astype(np.uint8)
    ax[0, k].imshow(img); ax[0, k].set_title(labels[base_eval.targets[i]]["raw"][:28], fontsize=9)
    ax[1, k].imshow(grey); ax[1, k].set_title("1. leaf found (background removed)", fontsize=9)
    ax[2, k].imshow(red); ax[2, k].set_title(f"2. spots: {lab_} ({frac:.0%})", fontsize=9)
    for a in ax[:, k]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/severity_examples.png", dpi=120); plt.close()
json.dump(SEV, open(f"{OUT_DIR}/severity_config.json", "w"), indent=1)

## 5 · Export to ExecuTorch, in a separate environment
This creates an isolated Python environment at `/tmp/et_env` with ExecuTorch and a matching CPU PyTorch, then runs `export_vision.py` inside it. The training kernel's PyTorch is never touched. The first run takes about 5–10 min to download packages.

In [ ]:
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"command failed ({r.returncode}): {cmd}"

PINS = "executorch==1.6.0 torch==2.15.0 torchvision==0.29.2 torchao==0.18.0"   # tested together
if not os.path.exists(f"{ENV_DIR}/bin/python"):
    sh(f"{sys.executable} -m pip install -q virtualenv")
    sh(f"{sys.executable} -m virtualenv -q {ENV_DIR}")
pip = f"{ENV_DIR}/bin/pip install -q --default-timeout=120 --retries 10"
# Prefer small CPU-only PyTorch wheels; fall back to the standard index (larger download) if that server fails.
if subprocess.run(f"{pip} --extra-index-url https://download.pytorch.org/whl/cpu {PINS}", shell=True).returncode != 0:
    print("CPU wheel index failed — retrying from the standard index")
    sh(f"{pip} {PINS}")
sh(f"{ENV_DIR}/bin/python -c \"import torch, executorch; print('export env: torch', torch.__version__)\"")

In [ ]:
EXPORT_SRC = r"""
import sys, os, json, time, argparse
import numpy as np, torch
from torch.export import export
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.runtime import Runtime

ap = argparse.ArgumentParser()
ap.add_argument("--work"); ap.add_argument("--out"); ap.add_argument("--n_eval", type=int, default=1500)
a = ap.parse_args()
sys.path.insert(0, a.work)
from vision_model import Classifier

sp = json.load(open(f"{a.out}/splits.json"))
ds = ImageFolder(sp["data_dir"], transform=T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()]))
assert ds.classes == sp["classes"], "class order mismatch"

m = Classifier(len(sp["classes"]), pretrained=False)
m.load_state_dict(torch.load(f"{a.out}/best.pt", map_location="cpu"))
m.export_mode = True; m.eval()
example = (torch.rand(1, 3, 224, 224),)

def to_pte(model, path):
    prog = to_edge_transform_and_lower(export(model, example), partitioner=[XnnpackPartitioner()]).to_executorch()
    open(path, "wb").write(prog.buffer)
    print(f"saved {path} ({len(prog.buffer)/1e6:.1f} MB)", flush=True)

to_pte(m, f"{a.out}/crop_disease_fp32.pte")

g = export(m, example).module()
prepared = prepare_pt2e(g, XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True)))
with torch.no_grad():
    for x, _ in DataLoader(Subset(ds, sp["train"][:300]), 32):
        for i in range(x.size(0)): prepared(x[i:i+1])          # exported graph is batch-1
to_pte(convert_pt2e(prepared), f"{a.out}/crop_disease_int8.pte")

res = {}
test = Subset(ds, sp["test"][:a.n_eval])
for name in ["fp32", "int8"]:
    meth = Runtime.get().load_program(f"{a.out}/crop_disease_{name}.pte").load_method("forward")
    ok, times = 0, []
    for x, y in DataLoader(test, 1):
        t = time.perf_counter(); p = meth.execute([x])[0]; times.append(time.perf_counter() - t)
        ok += int(p.argmax(1).item() == y.item())
    res[f"pte_{name}"] = {"accuracy": ok / len(test), "n_images": len(test),
                          "median_ms_notebook_cpu": 1000 * float(np.median(times)),
                          "size_mb": os.path.getsize(f"{a.out}/crop_disease_{name}.pte") / 1e6}
    print(name, res[f"pte_{name}"], flush=True)
json.dump(res, open(f"{a.out}/pte_metrics.json", "w"), indent=1)
"""
open(f"{WORK}/export_vision.py", "w").write(EXPORT_SRC)
sh(f"{ENV_DIR}/bin/python {WORK}/export_vision.py --work {WORK} --out {OUT_DIR} --n_eval {40 if FAST_TEST else 1500}")

metrics.update(json.load(open(f"{OUT_DIR}/pte_metrics.json")))
json.dump(metrics, open(f"{OUT_DIR}/metrics.json", "w"), indent=1, default=float)
print(json.dumps({k: v for k, v in metrics.items() if k != "per_class"}, indent=1, default=float))

## 6 · Package the outputs

In [ ]:
keep = ["crop_disease_int8.pte", "crop_disease_fp32.pte", "labels.json", "severity_config.json",
        "metrics.json", "confusion_matrix.png", "severity_examples.png"]
pkg = f"{WORK}/ps1_vision_outputs"
os.makedirs(pkg, exist_ok=True)
for f in keep: shutil.copy(f"{OUT_DIR}/{f}", pkg)
print("zipped →", shutil.make_archive(pkg, "zip", pkg))
print("INT8 test accuracy:", metrics["pte_int8"]["accuracy"], "· size", round(metrics["pte_int8"]["size_mb"], 1), "MB")